# LMSSPP Colab Live Backup Notebook

Runs the original Python widget classes.
This version includes a Colab-specific Plotly/FigureWidget compatibility setup.


In [ ]:
#@title Setup (Colab/local switch) { display-mode: "form" }
REPO_URL = "https://github.com/AdamSobieszek/LMSSPP.git" # @param {type:"string"}
REPO_BRANCH = "main" # @param {type:"string"}
CLONE_DIR = "/content/LMSSPP" # @param {type:"string"}
LOCAL_SRC_DIR = "/Users/adamsobieszek/pitch/pitch-website/public/notebooks/kuramoto/LMSSPP/src" # @param {type:"string"}

import os
import shutil
import subprocess
import sys
from pathlib import Path

# Flip: auto-detect Colab vs local notebook runtime.
IN_COLAB = "google.colab" in sys.modules
print("IN_COLAB =", IN_COLAB)

def run(cmd):
    print("+", " ".join(cmd))
    subprocess.check_call(cmd)

# Use notebook-friendly matplotlib mode in Colab branch.
if IN_COLAB:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().run_line_magic("cd", "/content")

    if Path(CLONE_DIR).exists():
        shutil.rmtree(CLONE_DIR)

    run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, CLONE_DIR])
    os.chdir(CLONE_DIR)

    # Colab + FigureWidget compatibility stack
    run([sys.executable, "-m", "pip", "install", "-U", "pip"])
    run([sys.executable, "-m", "pip", "install", "ipywidgets==7.7.1", "widgetsnbextension==3.6.10", "jupyterlab_widgets==1.1.11"])
    run([sys.executable, "-m", "pip", "install", "plotly==5.24.1", "anywidget>=0.9.13"])
    run([sys.executable, "-m", "pip", "install", "-e", "."])
    run([sys.executable, "-m", "pip", "install", "-e", ".[torch,live]"])

    from google.colab import output
    output.enable_custom_widget_manager()
    SRC_DIR = str(Path(CLONE_DIR) / "src")
else:
    # Local troubleshooting mode: do not clone/pull; use local source tree.
    src_path = Path(LOCAL_SRC_DIR).expanduser().resolve()
    if not src_path.exists():
        # Fallback: try to discover repo root from current working directory.
        cwd = Path.cwd().resolve()
        candidates = [cwd, *cwd.parents]
        found = None
        for c in candidates:
            if (c / "src" / "lmsspp").exists() and (c / "pyproject.toml").exists():
                found = c / "src"
                break
        if found is None:
            raise FileNotFoundError(f"Could not find local src path: {src_path}")
        src_path = found
    SRC_DIR = str(src_path)
    print("Using local SRC_DIR:", SRC_DIR)

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("Setup complete. cwd =", os.getcwd())
print("Using src:", SRC_DIR)


IN_COLAB = False
Using local SRC_DIR: /Users/adamsobieszek/pitch/pitch-website/public/notebooks/kuramoto/LMSSPP/src
Setup complete. cwd = /Users/adamsobieszek/pitch/pitch-website/public/notebooks/kuramoto/LMSSPP/notebooks
Using src: /Users/adamsobieszek/pitch/pitch-website/public/notebooks/kuramoto/LMSSPP/src


In [ ]:
# FigureWidget transport probe (run this before widget cells)
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

import plotly
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

print("IN_COLAB:", IN_COLAB)
print("plotly:", plotly.__version__)
print("ipywidgets:", widgets.__version__)

probe_fig = go.FigureWidget(go.Scatter(y=[1, 3, 2]))
probe_box = widgets.VBox([widgets.HTML("<b>FigureWidget probe</b>"), probe_fig])
display(probe_box)
print("If the mini chart above is visible, transport is working.")


IN_COLAB: False
plotly: 6.3.0
ipywidgets: 8.1.7


    'data': [{'type': 'scatter', 'uid':…

If the mini chart above is visible, transport is working.


In [ ]:
from IPython.display import display
from lmsspp.core.lms import (
    make_lms_ball3d_backward_two_sheet_widget,
    make_lms_ball3d_hydrodynamic_ensemble_widget,
    make_lms_ball3d_widget,
    make_lms_circle_plotly_widget,
)


## Circle Widget


In [ ]:
circle = make_lms_circle_plotly_widget(steps=300, dt=0.012, rng_seed=0)
display(circle.layout if hasattr(circle, "layout") else circle)


    'data': [{'hoverinfo': 'skip',
              'line': {'color': 'black', 'wid…

## Ball3D Widget


In [ ]:
ball3d = make_lms_ball3d_widget(rng_seed=0)
display(ball3d.layout if hasattr(ball3d, "layout") else ball3d)


Box(children=(FigureWidget({
    'data': [{'hoverinfo': 'skip',
              'line': {'color': 'lightgrey', '…

## Backward Two-Sheet Ball3D Widget


In [ ]:
backward = make_lms_ball3d_backward_two_sheet_widget(rng_seed=1)
display(backward.layout if hasattr(backward, "layout") else backward)


Box(children=(FigureWidget({
    'data': [{'hoverinfo': 'skip',
              'line': {'color': 'lightgrey', '…

## Hydrodynamic Ensemble Widget (lighter defaults for Colab)


In [ ]:
from lmsspp.lms_ball3d_widget import HYDRO_DEFAULT_CONTROLS, LMS3DControlSpec

light_hydro_controls = tuple(
    LMS3DControlSpec(
        key=spec.key,
        label=spec.label,
        min=spec.min,
        max=spec.max,
        step=spec.step,
        value=(600 if spec.key == "N" else (220 if spec.key == "steps" else spec.value)),
        integer=spec.integer,
        readout_format=spec.readout_format,
        continuous_update=spec.continuous_update,
    )
    for spec in HYDRO_DEFAULT_CONTROLS
)

hydro = make_lms_ball3d_hydrodynamic_ensemble_widget(
    controls=light_hydro_controls,
    rng_seed=2,
    display_points_cap=600,
)
display(hydro.layout if hasattr(hydro, "layout") else hydro)


Box(children=(FigureWidget({
    'data': [{'hoverinfo': 'skip',
              'line': {'color': 'lightgrey', '…